# ReaVector 3D bench — TripoSR in a clean Python 3.11 venv

Kaggle's default Python is 3.13 (torch 2.4 has no cp313 wheels). Build a clean 3.11
venv, install torch 2.4.1+cu121, build `torchmcubes --no-build-isolation`, then run
TripoSR on the T4 (GPU 1) and record time / peak VRAM / output.

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version || true
!gcc --version | head -1 || true
!python -V

In [ ]:
!git clone https://github.com/rizbiehelp-sys/reavectors.git
%cd reavectors
!pip -q install uv
!uv venv --python 3.11 --seed /tmp/venv
!/tmp/venv/bin/python -V

In [ ]:
!/tmp/venv/bin/pip -q install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!/tmp/venv/bin/pip -q install ninja
!/tmp/venv/bin/python -c "import torch;print('torch',torch.__version__,'cuda',torch.version.cuda,'avail',torch.cuda.is_available())"

In [ ]:
!TORCH_CUDA_ARCH_LIST="7.5" /tmp/venv/bin/pip install --no-build-isolation git+https://github.com/tatsy/torchmcubes.git 2>&1 | tail -25 || echo TORCHMCUBES_FAILED
!/tmp/venv/bin/python -c "import torchmcubes;print('torchmcubes OK')" || echo TORCHMCUBES_IMPORT_FAILED

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR /opt/TripoSR
!/tmp/venv/bin/pip -q install omegaconf trimesh xatlas huggingface_hub einops Pillow "numpy<2" potracer vtracer
!PYTHONPATH=/opt/TripoSR /tmp/venv/bin/python scripts/ai_bench.py --gpu 1 --models triposr --image examples/product.png 2>&1 | tail -30

In [ ]:
import glob, os
for f in sorted(glob.glob('bench/results/*')):
    if os.path.isfile(f):
        print('=====', f); print(open(f).read()[:4000])
